In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import numpy as np
import pyexcel_ods

In [ ]:
ods_file = snakemake.input.ods_file
existing_cap_file = snakemake.input.existing_cap
onshore_shapefile = snakemake.input.onshore_shapefile
offshore_shapefile = snakemake.input.offshore_shapefile

ods_output_file = snakemake.params.ods_output_file
year = int(snakemake.wildcards.year)

geodata_file = snakemake.output.geodata_file

existing_pcap_z_path = snakemake.output.existing_pcap_z_path
existing_pcap_r_path = snakemake.output.existing_pcap_r_path
gen_lim_z_path = snakemake.output.gen_lim_z_path


In [ ]:
# open ODS (Excel) file

excel_exist_pcap_z = pd.read_excel(
    ods_file,
    sheet_name="gen_exist_z",
    skiprows=0,
    engine="calamine",
)
excel_exist_pcap_r = pd.read_excel(
    ods_file,
    sheet_name="gen_exist_r",
    skiprows=0,
    engine="calamine",
)

excel_gen_lim_z = pd.read_excel(
    ods_file,
    sheet_name="gen_lim_z",
    skiprows=0,
    engine="calamine",
)

In [ ]:
# get shapefiles from intermediate data from highres WF

geodata_files = {
    "onshore": onshore_shapefile,
    "offshore_bottom": offshore_shapefile,
}

# Load onshore shapefile
euroshape = gpd.read_file(geodata_files["onshore"]).set_index(["index"])
aggr_countries = euroshape.CNTR_CODE.unique().tolist()  # get relevant countries
aggr_countries

In [ ]:
# Calculate boundaries based on onshore and offshore shapefiles
boundaries = []
for geodata_file_name, geodata_file_path in geodata_files.items():
    boundaries.append(gpd.read_file(geodata_file_path).set_index(["index"]))

boundaries = pd.concat(boundaries).bounds

boundaries = boundaries.groupby(lambda x: "bountry").agg(
    {"minx": "min", "miny": "min", "maxx": "max", "maxy": "max"}
)

boundaries

In [ ]:
exist_pcap_dataset = pd.read_csv(existing_cap_file)

# convert to geopandas
existing_pcap = (
    gpd.GeoDataFrame(
        exist_pcap_dataset,
        geometry=gpd.points_from_xy(exist_pcap_dataset.lon, exist_pcap_dataset.lat),
        crs="epsg:4326",
    )
    .loc[
        :,
        [
            "Technology",
            "status",
            "iso_code",
            "capacity_MW",
            "geometry",
            "commissioning_year",
            "estimated_retirement_year",
        ],
    ]
    .query("iso_code==@aggr_countries")  # select only relevant countries
)
existing_pcap

In [ ]:
# Clip/Remove plants outside the considered area

rectx1 = boundaries.loc["bountry", "minx"]
rectx2 = boundaries.loc["bountry", "maxx"]
recty1 = boundaries.loc["bountry", "miny"]
recty2 = boundaries.loc["bountry", "maxy"]


polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
)
existing_pcap = gpd.clip(existing_pcap, polygon)

In [ ]:
# Determine the nuts2 region for every power plant
all_techs = gpd.sjoin(euroshape, existing_pcap, how="right", predicate="contains").loc[
    :,
    [
        "Technology",
        "status",
        "index",
        "CNTR_CODE",
        "iso_code",
        "capacity_MW",
        "geometry",
        "commissioning_year",
        "estimated_retirement_year",
    ],
]

# Matched plants
no_error_techs = (
    all_techs.query("CNTR_CODE == iso_code")  # country codes match
    # conserve "geometry" for plotting (it could be removed)
    .loc[
        :,
        [
            "Technology",
            "status",
            "iso_code",
            "index",
            "capacity_MW",
            "commissioning_year",
            "estimated_retirement_year",
            "geometry",
        ],
    ]
    # .loc[:, ["Technology", "iso_code", "index", "capacity_MW","geometry"]]
)

# Mismatched plants (country codes mismatch)
error_techs = all_techs.query("CNTR_CODE != iso_code")


# Plants with different country assignment or nan assignment
# from shapefile and database
# There are two groups:
# 1) Plants on the coast and offshore
# 2) Plants on the border of two countries


## Plants on the coast and offshore
# Solution: assign closest region"europe_offshore.geojson",
offshore_plants = error_techs.query("CNTR_CODE.isna()").to_crs(3035)  # nan values

for idx, row in offshore_plants.iterrows():
    # Find the closest region
    ztemp = euroshape.loc[euroshape["CNTR_CODE"] == row["iso_code"], :]
    dists = ztemp.to_crs(3035).distance(row.geometry)
    if len(dists) == 0:
        break
    offshore_plants.loc[idx, "index"] = dists.idxmin()

offshore_plants = (
    offshore_plants.to_crs(4326)
    # conserve "geometry" for plotting
    .loc[
        :,
        [
            "Technology",
            "status",
            "iso_code",
            "index",
            "capacity_MW",
            "commissioning_year",
            "estimated_retirement_year",
            "geometry",
        ],
    ]
)

## Plants on the border of two countries
# Solution: Assign information from shape file
# Issues spotted in Hungary/Austria/Czech Republic
# Only 3 solar plants with 1m resolutiongem_nuclear
country_border_plants = (
    error_techs.query("~CNTR_CODE.isna()")  # different country assignment
)
country_border_plants = (
    country_border_plants
    # conserve "geometry" for plotting
    .loc[
        :,
        [
            "Technology",
            "status",
            "CNTR_CODE",
            "index",
            "capacity_MW",
            "commissioning_year",
            "estimated_retirement_year",
            "geometry",
        ],
    ].rename(columns={"CNTR_CODE": "iso_code"})
)

### Merge all the plants
merged_techs = pd.concat(
    [
        no_error_techs,
        offshore_plants,
        country_border_plants,
    ]
).reset_index(drop=True)


In [ ]:
if snakemake.output.existing_capacities_full != []:
    merged_techs.to_csv(snakemake.output.existing_capacities_full, index=False)

In [ ]:
# Filtering plants based on the lifetime of the decommissioned plants in GEM

merged_techs = merged_techs.query("(estimated_retirement_year>=@year)").loc[
    :, ["Technology", "iso_code", "index", "capacity_MW"]
]

In [ ]:
# Format the dataframe like in the ODS (Excel) file

# Technologies with finner resolution
r_techs = ["solar", "onshore", "offshore"]
exist_pcap_r = (
    merged_techs.query("Technology == @r_techs")
    .groupby(["Technology", "index", "iso_code"])
    .sum()
    .reset_index()
    .rename(columns={"capacity_MW": "value", "index": "region", "iso_code": "zone"})
    .assign(parameter="gen_exist_pcap_r")
    .assign(Year=year)
    .assign(limtype="FX")  # or FX
    .replace({"onshore": "Windonshore", "offshore": "Windoffshore", "solar": "Solar"})
    .reset_index(drop=True)
)

# All capacities aggregated to a coarser resolution
z_techs = ["solar", "onshore", "offshore", "nuclear"]
# z_techs = ["nuclear"]
exist_pcap_z = (
    merged_techs.query("Technology == @z_techs")
    .loc[:, ["Technology", "iso_code", "capacity_MW"]]
    .groupby(["Technology", "iso_code"])
    .sum()
    .reset_index()
    .pivot(index="Technology", columns="iso_code", values="capacity_MW")
    .reset_index("Technology")
    .assign(parameter="gen_exist_pcap_z")
    .assign(Year=year)
    .assign(limtype="FX")
    .replace(  # hardcoded - TODO: get name of technologies
        {
            "onshore": "Windonshore",
            "offshore": "Windoffshore",
            "solar": "Solar",
            "nuclear": "NuclearEPR",
        }
    )
)
exist_pcap_z

In [ ]:
## Update existing capacities based on gen lim z

zones_exist = [
    col
    for col in exist_pcap_z.columns
    if col not in ["Technology", "Year", "parameter", "limtype"]
]
technologies = exist_pcap_z.Technology.unique().tolist()

## Assign nan value to up lim and then fill it
exist_pcap_z_up = exist_pcap_z.copy()
exist_pcap_z_up.loc[:, zones_exist] = np.nan
exist_pcap_z_up["limtype"] = "UP"

for technology in technologies:
    row = excel_gen_lim_z.index[excel_gen_lim_z["Technology"] == technology]
    if len(row) == 1:
        for zone in zones_exist:
            col = excel_gen_lim_z.columns[excel_gen_lim_z.columns == zone]
            if len(col) == 1:
                gen_lim = excel_gen_lim_z.loc[row[0], col[0]]
                if gen_lim < float(
                    exist_pcap_z.loc[
                        exist_pcap_z.Technology == technology, zone
                    ].values[0]
                ):
                    exist_pcap_z_up.loc[
                        exist_pcap_z.Technology == technology, [zone]
                    ] = exist_pcap_z.loc[exist_pcap_z.Technology == technology, [zone]]
                    exist_pcap_z.loc[exist_pcap_z.Technology == technology, [zone]] = (
                        np.nan
                    )

exist_pcap_z_up = exist_pcap_z_up.dropna(subset=zones_exist, how="all")

exist_pcap_z = pd.concat(
    [
        exist_pcap_z,
        exist_pcap_z_up,
    ]
).dropna(subset=zones_exist, how="all")


exist_pcap_z

In [ ]:
# Combine/merge hydro and other techs

zones = [
    col
    for col in excel_gen_lim_z.columns
    if col not in ["Technology", "Year", "parameter", "limtype"]
]

combined_exist_pcap_z = pd.concat(
    [
        excel_exist_pcap_z,
        exist_pcap_z,
    ]
).loc[:, ["Technology", "Year", "parameter", "limtype"] + zones]

combined_exist_pcap_r = pd.concat(
    [
        excel_exist_pcap_r,
        exist_pcap_r,
    ]
)

combined_exist_pcap_z = combined_exist_pcap_z.replace({np.nan: ""})
combined_exist_pcap_r = combined_exist_pcap_r.replace({np.nan: ""})

combined_exist_pcap_z

In [ ]:
# Create csv files with the existing capacities by region (nuts2) and zone (country)

if not gen_lim_z_path == "":
    (combined_exist_pcap_z.to_csv(existing_pcap_z_path, index=False))
    (combined_exist_pcap_r.to_csv(existing_pcap_r_path, index=False))
    (excel_gen_lim_z.to_csv(gen_lim_z_path, index=False))


In [ ]:
# Load data
existing_data = pyexcel_ods.get_data(ods_file)


updated_dataframes = [combined_exist_pcap_z, combined_exist_pcap_r]
sheets = ["gen_exist_z", "gen_exist_r"]

for updated_dataframe, sheet in zip(updated_dataframes, sheets):
    # Update sheet
    existing_data[sheet] = updated_dataframe.values.tolist()  ## add values
    existing_data[sheet].insert(0, list(updated_dataframe.columns))  ## add headers

# Write again the file
pyexcel_ods.save_data(str(ods_output_file), existing_data)

In [ ]:
### Merge all the plants for plotting
techs_plot = pd.concat(
    [
        no_error_techs,
        offshore_plants,
        country_border_plants,
    ]
).reset_index(drop=True)
techs_plot = gpd.GeoDataFrame(
    techs_plot,
    crs="epsg:4326",
)
techs_plot.to_file(geodata_file)
